In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

In [23]:
# 残差块
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
        
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, X):
        
        Y = F.relu(self.bn1(self.conv1(X)))
        
        Y = self.bn2(self.conv2(Y))
        
        if self.conv3:
            
            X = self.conv3(X)
        
        return F.relu(Y + X)

In [25]:
class GlobalAvgPool2d(nn.Module):
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()
    def forward(self, x):
        return F.adaptive_avg_pool2d(x, (1, 1))

In [27]:
net = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
    
    nn.BatchNorm2d(64),
    
    nn.ReLU(),
    
    nn.MaxPool2d(kernel_size=2, stride=1, padding=1)
)


In [29]:
def resnet_block(in_channels, out_channels, num_residuals, first_block=False):
    
    if first_block:
        assert in_channels == out_channels  # 第一个模块的通道数与输入通道数一致
    
    blk = []
    
    for i in range(num_residuals):
        
        if i == 0 and not first_block:
            blk.append(Residual(in_channels, out_channels, use_1x1conv=True, stride=2))
        
        else:
            blk.append(Residual(out_channels, out_channels))
    
    return nn.Sequential(*blk)


In [32]:
net.add_module("resnet_block1", resnet_block(64, 64, 2, first_block=True))
net.add_module("resnet_block2", resnet_block(64, 128, 2))
net.add_module("resnet_block3", resnet_block(128, 256, 2))
net.add_module("resnet_block4", resnet_block(256, 512, 2))


In [34]:
net.add_module('global_avg_pool', GlobalAvgPool2d())
net.add_module('fc', nn.Sequential(nn.Flatten(), 
                                   nn.Linear(512, 10)
                                  )
              )

In [36]:
data_transform = transforms.Compose([
    transforms.Resize(256),     #resize图片
    transforms.CenterCrop(224), #随机裁剪
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) #标准化
])



In [38]:

train_sets = datasets.CIFAR10(root='cifar_10', 
                              train=True, 
                              download=True, 
                              transform=data_transform)

test_sets = datasets.CIFAR10(root='cifar_10', 
                             train=False, 
                             download=True, 
                             transform=data_transform)

batch_size = 64

train_loader = torch.utils.data.DataLoader(dataset=train_sets, 
                                           batch_size=batch_size, 
                                           shuffle=True)  #将数据打乱

test_loader = torch.utils.data.DataLoader(dataset=test_sets, 
                                          batch_size=batch_size, 
                                          shuffle=True)


In [40]:
def evaluate_accuracy(data_iter, net):
    acc_sum, n = 0.0, 0
    
    net.eval()
    for X, y in data_iter:
        acc_sum += (net(X).argmax(dim=1) == y).float().sum().item()
        
        n += y.shape[0]
    return acc_sum / n

In [42]:
def train(net, train_iter, test_iter, batch_size, optimizer, device, num_epochs):
    
    net = net.to(device)
    print("training on ", device)
    loss = torch.nn.CrossEntropyLoss()
    batch_count = 0
    for epoch in range(num_epochs):
        
        train_l_sum, train_acc_sum, n, start = 0.0, 0.0, 0, time.time()
        
        net.train()
        for X, y in train_iter:
            X = X.to(device)
            y = y.to(device)
            
            y_hat = net(X)
            l = loss(y_hat, y)
            
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
            
            train_l_sum += l.cpu().item()
            train_acc_sum += (y_hat.argmax(dim=1) == y).sum().cpu().item()
            n += y.shape[0]
            batch_count += 1

        test_acc = evaluate_accuracy(test_iter, net)
        print('epoch %d, loss %.4f, train acc %.3f, test acc %.3f, time %.1f sec'
              % (epoch + 1, train_l_sum / batch_count, train_acc_sum / batch_count, test_acc, time.time() - start))

In [44]:
torch.cuda.empty_cache()

In [46]:
num_epochs = 10

batch_size = 32

lr = 0.001

device = "cuda"

optimizer = torch.optim.Adam(net.parameters(), lr)

In [ ]:
#启动
train(net, train_loader, test_loader, batch_size, optimizer, device, num_epochs)

training on  cuda


### Densenet

In [48]:
def conv_block(in_channels, out_channels):
    blk = nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
    )
    return blk

In [50]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, in_channels, out_channels):
        super(DenseBlock, self).__init__()
        net = []
        for i in range(num_convs):
            in_c = in_channels + i * out_channels
            net.append(conv_block(in_c, out_channels))
        self.net = nn.ModuleList(net)
        self.out_channels = in_channels + num_convs * out_channels  # 计算输出通道数

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            X = torch.cat((X, Y), dim=1)  # 在通道维上将输入和输出连结
        return X

In [52]:
def transition_block(in_channels, out_channels):
    blk = nn.Sequential(
        nn.BatchNorm2d(in_channels),
        nn.ReLU(),
        nn.Conv2d(in_channels, out_channels, kernel_size=1),
        nn.AvgPool2d(kernel_size=2, stride=2)
    )
    return blk

In [54]:
net = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

num_channels, growth_rate = 64, 32
num_convs_in_dense_blocks = [4, 4, 4, 4]

In [56]:
for i, num_convs in enumerate(num_convs_in_dense_blocks):
    DB = DenseBlock(num_convs, num_channels, growth_rate)
    net.add_module(f"DenseBlock_{i}", DB)
    num_channels = DB.out_channels
    if i != len(num_convs_in_dense_blocks) - 1:
        net.add_module(f"transition_block_{i}", transition_block(num_channels, num_channels // 2))
        num_channels = num_channels // 2

In [58]:
net.add_module("BN", nn.BatchNorm2d(num_channels))
net.add_module("relu", nn.ReLU())
net.add_module("global_avg_pool", GlobalAvgPool2d())
net.add_module("fc", nn.Sequential(nn.Flatten(), nn.Linear(num_channels, 10)))

In [60]:
batch_size = 256


device = 'cuda'
lr, num_epochs = 0.001, 5
optimizer = torch.optim.Adam(net.parameters(), lr=lr)


In [62]:
#启动
train(net, train_loader, test_loader, batch_size, optimizer, device, num_epochs)

training on  cuda


KeyboardInterrupt: 